# Multistream training (self-contained)

All logic lives in this notebook: **no imports from** `train_multistream_cnn.py`, `train_multistream_cnn_rot.py`, `train_fusion_multistream_cnn.py`, or `train_fusion_tof_multistream_cnn_finetune.py`.

## What runs

1. Accelerometer multistream CNN → `models/deep_learning_models/Multistream_CNN_acc_only.pth`
2. Rotation multistream CNN → `Multistream_CNN_rot_only.pth`
3. Acc+rot fusion → `Fusion_Multistream_CNN.pth`
4. ToF CNN (skipped if `ToF_CNN.pth` already exists) → `ToF_CNN.pth`
5. Fusion+ToF fine-tune → `Fusion_ToF_Multistream_CNN_finetune.pth`

Optional warm-start: if `Fusion_ToF_Multistream_CNN.pth` exists, compatible weights are loaded before fine-tuning.

## Practices used

- **Configuration** at the top of the code cell (paths, batch size, learning rates).
- **One `fit_model`** for all stages; small `train_step` / `eval_step` pairs per modality to avoid copy-pasted loops.
- **One `save_model_and_metadata`** for full checkpoints.

## Requirements

`./data/train.csv` or KaggleHub credentials. Long runtimes expected.

Restart the kernel after editing this notebook’s code, then run all cells top to bottom.


## 1. Initialization and Data Preparation

Set up all necessary imports, configure environment, define constants, and load/split/encode data.

Includes:
- PyTorch and data processing libraries
- Seed values for reproducibility
- Repository root path detection
- Display options
- Global configuration constants
- Data loading, splitting (subject-level), and label encoding functions
- **Execute immediately**: load data, split train/val/test, encode gesture labels


In [12]:
from __future__ import annotations

import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from torch import nn, optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# ============================================================================
# PART 1: Repository Root and Display Configuration
# ============================================================================

# Repo root: cwd or parents must contain this marker
MARKER = "train_multistream_cnn.py"


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / MARKER).is_file():
            return d
    raise FileNotFoundError(
        f"Could not find {MARKER} from {start}. cd to the sadna repo or open the notebook from there."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

print(f"✓ Repo root: {REPO_ROOT}")
print(f"✓ Working directory: {os.getcwd()}")

# Random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Configure pandas display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

print(f"✓ Random seeds set: {RANDOM_STATE}")
print(f"✓ Pandas display options configured")

# ============================================================================
# PART 2: Global Configuration Constants
# ============================================================================

# Gesture classes - BFRB (Body-Focused Repetitive Behavior) gestures
BFRB_GESTURES = [
    "Above ear - pull hair",
    "Eyebrow - pull hair",
    "Eyelash - pull hair",
    "Forehead - pull hairline",
    "Forehead - scratch",
    "Cheek - pinch skin",
    "Neck - pinch skin",
    "Neck - scratch",
]

# Sensor columns
ROT_COLS = ["rot_x", "rot_y", "rot_z", "rot_w"]
TOF_MIN, TOF_MAX = 0.0, 249.0

# Model architecture dimensions
TRAIN_MODALITY_DROPOUT_ROT = 0.1
TRAIN_MODALITY_DROPOUT_TOF = 0.1
ROT_FEAT_DIM = 64
TOF_FEAT_DIM = 128
FUSION_INERTIAL_DIM = 128

# Fine-tuning head configuration
HEAD_HIDDEN_DIMS = (256, 128)
HEAD_DROPOUT = 0.2
LR_HEAD_FT = 1e-3
LR_BACKBONE_FT = 1e-5
WEIGHT_DECAY_FT = 1e-4

# Model checkpoint paths
PATH_ACC = "models/deep_learning_models/Multistream_CNN_acc_only.pth"
PATH_ROT = "models/deep_learning_models/Multistream_CNN_rot_only.pth"
PATH_FUSION = "models/deep_learning_models/Fusion_Multistream_CNN.pth"
PATH_TOF = "models/deep_learning_models/ToF_CNN.pth"
PATH_TRIPLE_WARMSTART = "models/deep_learning_models/Fusion_ToF_Multistream_CNN.pth"
PATH_FINETUNE = "models/deep_learning_models/Fusion_ToF_Multistream_CNN_finetune.pth"
PATH_HELIOS = "models/deep_learning_models/from_notebook/HeliosInertialNet.pth"

# Training hyperparameters
BATCH_SIZE = 16
NUM_EPOCHS = 100
EARLY_STOP_PATIENCE = 15
LR = 0.001
WEIGHT_DECAY = 1e-4
SCHED_FACTOR = 0.8
SCHED_PATIENCE = 10

print("✓ All configuration constants loaded")

# ============================================================================
# PART 3: Data Loading Functions
# ============================================================================

def load_train_data() -> pd.DataFrame:
    """Load training data from ./data/train.csv or KaggleHub."""
    try:
        path = "./data"
        train_df = pd.read_csv(os.path.join(path, "train.csv"))
        print(f"✓ Loaded train.csv from {path}")
    except FileNotFoundError:
        try:
            import kagglehub

            os.environ["KAGGLE_CONFIG_DIR"] = os.path.expanduser(".kaggle")
            path = kagglehub.competition_download("cmi-detect-behavior-with-sensor-data")
            train_df = pd.read_csv(os.path.join(path, "train.csv"))
            print(f"✓ Loaded train.csv via kagglehub from {path}")
        except Exception as e:
            raise FileNotFoundError(
                "Neither ./data/train.csv nor kagglehub data found."
            ) from e
    return train_df

print("✓ load_train_data() defined")

# ============================================================================
# PART 4: Data Splitting (Subject-Level Stratification)
# ============================================================================

def final_robust_split(
    df: pd.DataFrame,
    subject_col: str = "subject",
    bfrb_col: str = "is_bfrb",
    test_size: float = 0.15,
    val_size: float = 0.15,
):
    """
    Subject-level stratified split. Groups subjects by BFRB proportion.
    Returns (train_df, val_df, test_df).
    """
    sub_logic = df.groupby(subject_col)[bfrb_col].mean().reset_index()
    sub_logic["broad_cat"] = (sub_logic[bfrb_col] > 0.5).astype(int)
    unique_subs = sub_logic.sample(frac=1, random_state=RANDOM_STATE)
    n_total = len(unique_subs)
    n_test = int(n_total * test_size)
    n_val = int(n_total * val_size)
    test_subs = unique_subs.iloc[:n_test][subject_col]
    val_subs = unique_subs.iloc[n_test : n_test + n_val][subject_col]
    train_subs = unique_subs.iloc[n_test + n_val :][subject_col]
    return (
        df[df[subject_col].isin(train_subs)],
        df[df[subject_col].isin(val_subs)],
        df[df[subject_col].isin(test_subs)],
    )

print("✓ final_robust_split() defined")

# ============================================================================
# PART 5: Label Encoding
# ============================================================================

def apply_label_encoding(train_df, trainset_df, valset_df, testset_df):
    """Fit LabelEncoder on training gestures, map all sets, return encoded dfs."""
    le = LabelEncoder()
    le.fit(train_df["gesture"].unique())
    gesture_map = dict(zip(le.classes_, le.transform(le.classes_)))
    
    train_df = train_df.copy()
    train_df["gesture_encoded"] = train_df["gesture"].map(gesture_map)
    trainset_df = trainset_df.copy()
    trainset_df["gesture_encoded"] = trainset_df["gesture"].map(gesture_map)
    valset_df = valset_df.copy()
    valset_df["gesture_encoded"] = valset_df["gesture"].map(gesture_map)
    testset_df = testset_df.copy()
    testset_df["gesture_encoded"] = testset_df["gesture"].map(gesture_map)
    
    return train_df, trainset_df, valset_df, testset_df, le, gesture_map

print("✓ apply_label_encoding() defined")

# ============================================================================
# PART 6: Data Preparation Helper
# ============================================================================

def load_split_data_encoded():
    """Load CSV, add BFRB flag, split into train/val/test, encode labels."""
    train_df = load_train_data()
    train_df["is_bfrb"] = train_df["gesture"].isin(BFRB_GESTURES)
    trainset_df, valset_df, testset_df = final_robust_split(train_df)
    trainset_df = trainset_df.reset_index(drop=True)
    valset_df = valset_df.reset_index(drop=True)
    testset_df = testset_df.reset_index(drop=True)
    return apply_label_encoding(train_df, trainset_df, valset_df, testset_df)

print("✓ load_split_data_encoded() defined")

# ============================================================================
# PART 7: Execute Data Loading and Splitting
# ============================================================================

train_df_full, trainset_df, valset_df, testset_df, le, gesture_map = load_split_data_encoded()
print(f"✓ Data split: train {len(trainset_df)} rows ({trainset_df['sequence_id'].nunique()} seq), "
      f"val {len(valset_df)} ({valset_df['sequence_id'].nunique()} seq), "
      f"test {len(testset_df)} ({testset_df['sequence_id'].nunique()} seq)")


✓ Repo root: /Users/eldadlivschitz/Documents/study/data_science_workshop_20936/sadna_git_env/sadna
✓ Working directory: /Users/eldadlivschitz/Documents/study/data_science_workshop_20936/sadna_git_env/sadna
✓ Random seeds set: 42
✓ Pandas display options configured
✓ All configuration constants loaded
✓ load_train_data() defined
✓ final_robust_split() defined
✓ apply_label_encoding() defined
✓ load_split_data_encoded() defined
✓ Loaded train.csv from ./data


/var/folders/80/npxmtc2j0qjbb4wm1sm23l_00000gp/T/ipykernel_44521/3520883552.py:195: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["is_bfrb"] = train_df["gesture"].isin(BFRB_GESTURES)


✓ Data split: train 406777 rows (5762 seq), val 79383 (1170 seq), test 88785 (1219 seq)


## 2. Initial Deep Learning Model - Accelerometer Only

Baseline model using 1D CNN + Bidirectional LSTM architecture (HeliosInertialNet) trained on accelerometer data only.
This serves as the first deep learning approach before advancing to more complex multi-modal fusion models.

Architecture: Conv1d (3 layers) → Bidirectional LSTM → Global pooling → FC head with dropout

In [10]:
# ============================================================================
# SECTION 2: HELIOS INERTIAL NET - ACCELEROMETER ONLY BASELINE
# ============================================================================

class MinMaxNormalize(nn.Module):
    """Normalize input to [0, 1] range using given min/max bounds."""
    def __init__(self, min_val: float, max_val: float):
        super().__init__()
        self.min_val = min_val
        self.max_val = max_val

    def forward(self, x):
        x = torch.clamp(x, self.min_val, self.max_val)
        return (x - self.min_val) / (self.max_val - self.min_val + 1e-6)


class HeliosInertialNet(nn.Module):
    """Initial baseline model: Conv + LSTM + FC for accelerometer data only."""
    def __init__(self, num_classes: int = 18):
        super().__init__()
        self.conv1 = nn.Sequential(
            MinMaxNormalize(-20.0, 20.0),
            nn.Conv1d(3, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
            bidirectional=True,
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, acc):
        x = self.conv1(acc)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.permute(0, 2, 1)  # (B, T, 256)
        x, _ = self.lstm(x)      # (B, T, 256) - bidirectional
        x = x.permute(0, 2, 1)  # (B, 256, T)
        x_avg = self.global_avg_pool(x).squeeze(-1)
        x_max = self.global_max_pool(x).squeeze(-1)
        x = torch.cat([x_avg, x_max], dim=1)  # (B, 512)
        return self.fc(x)


class InertialSequenceDataset(Dataset):
    """Dataset for accelerometer-only sequences."""
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        acc = torch.tensor(data[["acc_x", "acc_y", "acc_z"]].values, dtype=torch.float32).T
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return acc, label


def collate_inertial(batch):
    """Pad acc sequences to same length, transpose to (B, 3, T)."""
    accs, labels = zip(*batch)
    acc_padded = pad_sequence([a.T for a in accs], batch_first=True, padding_value=0).transpose(1, 2)
    return acc_padded, torch.stack(labels)


def ts_acc(m, batch, device, crit):
    """Train step: accelerometer only."""
    acc, y = batch
    acc = acc.to(device)
    y = y.to(device)
    out = m(acc)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_acc(m, batch, device, crit):
    """Eval step: accelerometer only."""
    acc, y = batch
    acc = acc.to(device)
    y = y.to(device)
    out = m(acc)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()


def train_helios_inertial(device: torch.device):
    """Train HeliosInertialNet on accelerometer data only."""
    print("\n" + "=" * 80)
    print("HELIOS INERTIAL NET: Initial Accelerometer-Only Model")
    print("=" * 80)
    
    n_cls = len(le.classes_)
    train_seq_ids = trainset_df["sequence_id"].unique().tolist()
    val_seq_ids = valset_df["sequence_id"].unique().tolist()
    test_seq_ids = testset_df["sequence_id"].unique().tolist()
    
    tl = DataLoader(
        InertialSequenceDataset(trainset_df),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_inertial,
    )
    vl = DataLoader(
        InertialSequenceDataset(valset_df),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_inertial,
    )
    
    model = HeliosInertialNet(num_classes=n_cls).to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    
    hist = fit_model(model, tl, vl, device, crit, opt, sch, PATH_HELIOS, ts_acc, es_acc)
    
    # Load best checkpoint
    ckpt = torch.load(PATH_HELIOS, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    
    # Save with metadata
    save_model_and_metadata(model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_HELIOS)
    
    print(f"✓ HeliosInertialNet training complete")
    return model, hist, le


print("✓ HeliosInertialNet architecture and dataset defined")
print("✓ train_helios_inertial() function defined (will execute after Section 3+ utility functions)")


✓ HeliosInertialNet architecture and dataset defined
✓ train_helios_inertial() function defined (will execute after Section 3+ utility functions)


## 3. Quaternion (Rotation) Data Cleaning

Remove sequences with missing or invalid quaternion (rotation) values to ensure data quality for rotation models.


In [ ]:
def drop_sequences_with_missing_quaternions(df: pd.DataFrame, rot_cols=None):
    """Remove sequences that have any NaN values in rotation columns."""
    if rot_cols is None:
        rot_cols = ROT_COLS
    seq_has_missing = df.groupby("sequence_id")[rot_cols].apply(lambda g: g.isna().any().any())
    valid_seq_ids = seq_has_missing[~seq_has_missing].index.tolist()
    return df[df["sequence_id"].isin(valid_seq_ids)].reset_index(drop=True), valid_seq_ids

print("✓ drop_sequences_with_missing_quaternions() defined")

## 4. Time-of-Flight (ToF) Data Processing

Handle ToF sensor data: extract columns, validate data, clean missing values, normalize, and reshape to spatial images.


In [ ]:
def get_tof_columns(df: pd.DataFrame):
    """Extract all ToF column names (prefixed with 'tof_')."""
    return [c for c in df.columns if c.startswith("tof_")]


def drop_sequences_with_missing_tof(df: pd.DataFrame, tof_cols=None):
    """Remove sequences with no valid ToF data (all values are -1)."""
    if tof_cols is None:
        tof_cols = get_tof_columns(df)
    seq_has_any_valid = df.groupby("sequence_id")[tof_cols].apply(lambda g: (g != -1).any().any())
    valid_seq_ids = seq_has_any_valid[seq_has_any_valid].index.tolist()
    return df[df["sequence_id"].isin(valid_seq_ids)].reset_index(drop=True), valid_seq_ids


def _tof_frame_to_5x8x8(tof_flat: np.ndarray) -> np.ndarray:
    """Reshape flat ToF array to (batch, 5, 8, 8) spatial image."""
    if tof_flat.ndim == 1:
        tof_flat = tof_flat.reshape(1, -1)
    t = tof_flat.shape[0]
    x = tof_flat.reshape(t, 5, 64).reshape(t, 5, 8, 8)
    return x


def _normalize_tof_01(x: np.ndarray) -> np.ndarray:
    """Normalize ToF values to [0, 1] range with robust handling of missing values."""
    x = np.asarray(x, dtype=np.float64)
    x = np.nan_to_num(x, nan=TOF_MAX, posinf=TOF_MAX, neginf=TOF_MIN)
    x = np.clip(x, TOF_MIN, TOF_MAX)
    denom = float(TOF_MAX - TOF_MIN) or 1.0
    out = (x - TOF_MIN) / denom
    out = np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)
    return np.clip(out, 0.0, 1.0)


def compute_sequence_sensor_stats(df: pd.DataFrame, tof_cols, split_name: str = ""):
    """Print summary of sensor data availability per split."""
    n_seq = df["sequence_id"].nunique()
    if n_seq == 0:
        print(f"=== {split_name}: no sequences ===")
        return

    def _miss_rot(g):
        return g[ROT_COLS].isna().any().any()

    def _miss_tof(g):
        return not (g[tof_cols] != -1).any().any()

    gr = df.groupby("sequence_id", sort=False)
    miss_rot = gr.apply(_miss_rot)
    miss_tof = gr.apply(_miss_tof)
    mr = miss_rot.astype(bool)
    mt = miss_tof.astype(bool)
    title = split_name or "split"
    print(f"\n--- Sequence sensor availability: {title} ({n_seq} sequences) ---")
    print(f"  Missing ROT: {int(mr.sum())} ({100 * mr.mean():.2f}%)")
    print(f"  Missing ToF: {int(mt.sum())} ({100 * mt.mean():.2f}%)")

print("✓ ToF data processing functions defined")

## 5. Model Architectures - Part 1: Normalization and Single-Modality CNNs

Define neural network architectures: MinMaxNormalize layer, accelerometer-only CNN, and rotation-only CNN.


In [ ]:
class MinMaxNormalize(nn.Module):
    """Normalize input to [0, 1] range using given min/max bounds."""
    def __init__(self, min_val: float, max_val: float):
        super().__init__()
        self.min_val = min_val
        self.max_val = max_val

    def forward(self, x):
        x = torch.clamp(x, self.min_val, self.max_val)
        return (x - self.min_val) / (self.max_val - self.min_val + 1e-6)


class MultistreamCNNInertialNet(nn.Module):
    """CNN for accelerometer (inertial) data only."""
    def __init__(self, num_classes: int = 18):
        super().__init__()
        self.norm = MinMaxNormalize(-20.0, 20.0)
        self.conv1_block = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=4, stride=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=6, stride=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.max_pool = nn.MaxPool1d(kernel_size=2, stride=1)
        self.adaptive_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, acc):
        x = self.norm(acc)
        x = self.conv1_block(x)
        x = self.conv2_block(x)
        x = self.max_pool(x)
        x = self.adaptive_avg_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class MultistreamCNNRotNet(nn.Module):
    """CNN for rotation (quaternion) data only."""
    def __init__(self, num_classes: int = 18):
        super().__init__()
        self.norm = MinMaxNormalize(-1.0, 1.0)
        self.conv1_block = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=4, stride=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=6, stride=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.max_pool = nn.MaxPool1d(kernel_size=2, stride=1)
        self.adaptive_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, rot):
        x = self.norm(rot)
        x = self.conv1_block(x)
        x = self.conv2_block(x)
        x = self.max_pool(x)
        x = self.adaptive_avg_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

print("✓ Model architectures Part 1 defined")

## 6. Model Architectures - Part 2: Fusion and ToF Models

Define multi-modal fusion models and Time-of-Flight (ToF) 2D CNN architecture.


In [ ]:
class FusionMultistreamCNN(nn.Module):
    """Fuses accelerometer and rotation features."""
    def __init__(self, acc_backbone, rot_backbone, num_classes: int = 18, hidden_dim: int = 64):
        super().__init__()
        self.acc_backbone = acc_backbone
        self.rot_backbone = rot_backbone
        self.fusion_classifier = nn.Sequential(
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, acc, rot):
        acc_feat = self.acc_backbone(acc)
        rot_feat = self.rot_backbone(rot)
        return self.fusion_classifier(torch.cat([acc_feat, rot_feat], dim=1))


def fuse_pairwise(x):
    """Fuse pairs of channels by summing them (reduces 5x8x8 to expected spatial dims)."""
    b, c, h, w = x.shape
    x = x.view(b, c // 2, 2, h, w)
    return x.sum(dim=2)


TOF_CNN_FEAT_SIZE = 384


class ToFCNN(nn.Module):
    """2D CNN for Time-of-Flight depth images."""
    def __init__(self, num_classes: int = 18, dropout_p: float = 0.1):
        super().__init__()
        self.dropout_p = dropout_p
        self.conv1 = nn.Sequential(
            nn.Conv2d(5, 8, kernel_size=(2, 4), stride=(2, 2)),
            nn.BatchNorm2d(8),
            nn.PReLU(num_parameters=8),
            nn.Dropout(p=dropout_p),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(4, 16, kernel_size=2, stride=1),
            nn.BatchNorm2d(16),
            nn.PReLU(num_parameters=16),
            nn.Dropout(p=dropout_p),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=2, stride=1, padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.PReLU(num_parameters=32),
            nn.Dropout(p=dropout_p),
        )
        self.classifier = nn.Sequential(
            nn.Linear(TOF_CNN_FEAT_SIZE, 128),
            nn.PReLU(num_parameters=128),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, 100),
            nn.PReLU(num_parameters=100),
            nn.Dropout(p=dropout_p),
            nn.Linear(100, num_classes),
        )

    def forward(self, tof_padded, lengths):
        tof_padded = torch.clamp(tof_padded, 0.0, 1.0)
        lengths = lengths.clamp(min=1)
        b, max_t, c, h, w = tof_padded.shape
        x = tof_padded.view(b * max_t, c, h, w)
        x = self.conv1(x)
        x = fuse_pairwise(x)
        x = self.conv2(x)
        x = fuse_pairwise(x)
        x = self.conv3(x)
        x = torch.flatten(x, 1)
        x = x.view(b, max_t, -1)
        mask = torch.arange(max_t, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(x.dtype)
        lc = lengths.unsqueeze(1).clamp(min=1).to(x.dtype)
        x = (x * mask).sum(dim=1) / lc
        return self.classifier(x)

print("✓ Model architectures Part 2 defined")

## 7. Model Architectures - Part 3: Fine-tuning Models

Define models for fine-tuning: partial backbone freeze, and complete ToF+Fusion deep model.


In [ ]:
def make_deep_fusion_head(in_dim: int, num_classes: int, hidden_dims, dropout_p: float) -> nn.Sequential:
    """Build a deep fusion head with LayerNorm and ReLU activations."""
    layers = []
    d_in = in_dim
    for d_out in hidden_dims:
        layers.extend(
            [
                nn.Linear(d_in, d_out),
                nn.LayerNorm(d_out),
                nn.ReLU(),
                nn.Dropout(p=dropout_p),
            ]
        )
        d_in = d_out
    layers.append(nn.Linear(d_in, num_classes))
    return nn.Sequential(*layers)


class FusionPartialUnfreeze128(nn.Module):
    """Wrapper that freezes early layers of acc/rot backbones, unfreezes later parts."""
    def __init__(self, fusion_model: FusionMultistreamCNN):
        super().__init__()
        self.acc_backbone = fusion_model.acc_backbone
        self.rot_backbone = fusion_model.rot_backbone
        for backbone in (self.acc_backbone, self.rot_backbone):
            for p in backbone.norm.parameters():
                p.requires_grad = False
            for p in backbone.conv1_block.parameters():
                p.requires_grad = False
            for p in backbone.conv2_block.parameters():
                p.requires_grad = True
            for p in backbone.classifier.parameters():
                p.requires_grad = True

    def forward(self, acc, rot):
        return torch.cat([self.acc_backbone(acc), self.rot_backbone(rot)], dim=1)


class ToFFusionMultistreamCNNDeepFinetune(nn.Module):
    """Complete fine-tuning model: Fusion (128-dim) + ToF features + deep head, with missing modality handling."""
    def __init__(self, fusion_128, tof_feat, num_classes: int = 18):
        super().__init__()
        self.fusion_128 = fusion_128
        self.tof_feat = tof_feat
        self.missing_rot = nn.Parameter(torch.zeros(ROT_FEAT_DIM))
        self.missing_tof = nn.Parameter(torch.zeros(TOF_FEAT_DIM))
        in_dim = FUSION_INERTIAL_DIM + TOF_FEAT_DIM
        self.fusion_classifier = make_deep_fusion_head(
            in_dim, num_classes, HEAD_HIDDEN_DIMS, HEAD_DROPOUT
        )
        self.train_dropout_rot = TRAIN_MODALITY_DROPOUT_ROT
        self.train_dropout_tof = TRAIN_MODALITY_DROPOUT_TOF

    def forward(self, acc, rot, tof_padded, lengths, has_rot, has_tof):
        b = acc.shape[0]
        device = acc.device
        dtype = acc.dtype
        has_rot = has_rot.to(device=device, dtype=torch.bool)
        has_tof = has_tof.to(device=device, dtype=torch.bool)
        
        # Training-time modality dropout for robustness
        if self.training and (self.train_dropout_rot > 0 or self.train_dropout_tof > 0):
            if self.train_dropout_rot > 0:
                drop = (torch.rand(b, device=device) < self.train_dropout_rot) & has_rot
                has_rot = has_rot & ~drop
            if self.train_dropout_tof > 0:
                drop = (torch.rand(b, device=device) < self.train_dropout_tof) & has_tof
                has_tof = has_tof & ~drop
        
        # Acc+Rot fusion features
        acc_feat = self.fusion_128.acc_backbone(acc)
        rot_feat = torch.zeros(b, ROT_FEAT_DIM, device=device, dtype=dtype)
        if has_rot.any():
            idx = has_rot.nonzero(as_tuple=True)[0]
            rot_feat[idx] = self.fusion_128.rot_backbone(rot[idx])
        if (~has_rot).any():
            idx = (~has_rot).nonzero(as_tuple=True)[0]
            rot_feat[idx] = self.missing_rot.unsqueeze(0).expand(len(idx), -1)
        fr = torch.cat([acc_feat, rot_feat], dim=1)
        
        # ToF features
        tf = torch.zeros(b, TOF_FEAT_DIM, device=device, dtype=dtype)
        if has_tof.any():
            idx = has_tof.nonzero(as_tuple=True)[0]
            tf[idx] = self.tof_feat(tof_padded[idx], lengths[idx])
        if (~has_tof).any():
            idx = (~has_tof).nonzero(as_tuple=True)[0]
            tf[idx] = self.missing_tof.unsqueeze(0).expand(len(idx), -1)
        
        return self.fusion_classifier(torch.cat([fr, tf], dim=1))

print("✓ Model architectures Part 3 defined")

## 8. Checkpoint Loading and Management

Load checkpoints, extract feature extractors, build parameter groups, and save model states.


In [ ]:
def load_state_dict_into_model(model: nn.Module, checkpoint_path: str, device: torch.device):
    """Load checkpoint (dict or state_dict directly) into model."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)


def make_acc_feature_extractor(num_classes: int, checkpoint_path: str, device: torch.device):
    """Load acc backbone and freeze all parameters, use only first 4 classifier layers."""
    model = MultistreamCNNInertialNet(num_classes=num_classes)
    load_state_dict_into_model(model, checkpoint_path, device)
    model.classifier = nn.Sequential(
        model.classifier[0],
        model.classifier[1],
        model.classifier[2],
        model.classifier[3],
    )
    for p in model.parameters():
        p.requires_grad = False
    return model.to(device)


def make_rot_feature_extractor(num_classes: int, checkpoint_path: str, device: torch.device):
    """Load rot backbone and freeze all parameters, use only first 4 classifier layers."""
    model = MultistreamCNNRotNet(num_classes=num_classes)
    load_state_dict_into_model(model, checkpoint_path, device)
    model.classifier = nn.Sequential(
        model.classifier[0],
        model.classifier[1],
        model.classifier[2],
        model.classifier[3],
    )
    for p in model.parameters():
        p.requires_grad = False
    return model.to(device)


def make_fusion_finetune_128(num_classes: int, checkpoint_path: str, device: torch.device):
    """Load fusion model and wrap in partial unfreeze (fine-tune middle/late layers)."""
    acc_backbone = MultistreamCNNInertialNet(num_classes=num_classes)
    acc_backbone.classifier = nn.Sequential(
        acc_backbone.classifier[0],
        acc_backbone.classifier[1],
        acc_backbone.classifier[2],
        acc_backbone.classifier[3],
    )
    rot_backbone = MultistreamCNNRotNet(num_classes=num_classes)
    rot_backbone.classifier = nn.Sequential(
        rot_backbone.classifier[0],
        rot_backbone.classifier[1],
        rot_backbone.classifier[2],
        rot_backbone.classifier[3],
    )
    fusion = FusionMultistreamCNN(acc_backbone, rot_backbone, num_classes=num_classes, hidden_dim=64)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    fusion.load_state_dict(state)
    return FusionPartialUnfreeze128(fusion).to(device)


def make_tof_finetune_extractor(num_classes: int, checkpoint_path: str, device: torch.device, dropout_p: float = 0.1):
    """Load ToF model, freeze early layers, unfreeze late layers and classifier."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
        nc = ckpt.get("num_classes", num_classes)
        dropout_p = ckpt.get("dropout_p", dropout_p)
    else:
        sd = ckpt
        nc = num_classes
    tof_model = ToFCNN(num_classes=nc, dropout_p=dropout_p)
    tof_model.load_state_dict(sd)
    tof_model.classifier = nn.Sequential(tof_model.classifier[0], tof_model.classifier[1])
    for p in tof_model.conv1.parameters():
        p.requires_grad = False
    for p in tof_model.conv2.parameters():
        p.requires_grad = False
    for p in tof_model.conv3.parameters():
        p.requires_grad = True
    for p in tof_model.classifier.parameters():
        p.requires_grad = True
    return tof_model.to(device)


def load_state_dict_shape_safe(model: nn.Module, state_dict: dict):
    """Load state dict, skipping layers with shape mismatch (e.g., warm-start with different head)."""
    target = model.state_dict()
    to_load = {}
    skipped = []
    for k, v in state_dict.items():
        if k not in target:
            continue
        if target[k].shape != v.shape:
            skipped.append(k)
            continue
        to_load[k] = v
    missing = [k for k in target if k not in to_load]
    model.load_state_dict(to_load, strict=False)
    return list(to_load.keys()), skipped, missing


def collect_param_groups(model: ToFFusionMultistreamCNNDeepFinetune):
    """Collect trainable params: head, fusion backbone, tof backbone."""
    head_params = list(model.fusion_classifier.parameters()) + [model.missing_rot, model.missing_tof]
    fusion_bb = []
    for backbone in (model.fusion_128.acc_backbone, model.fusion_128.rot_backbone):
        fusion_bb.extend([p for p in backbone.parameters() if p.requires_grad])
    tof_params = [p for p in model.tof_feat.parameters() if p.requires_grad]
    head_ids = {id(p) for p in head_params}
    fusion_bb = [p for p in fusion_bb if id(p) not in head_ids]
    tof_params = [p for p in tof_params if id(p) not in head_ids]
    return head_params, fusion_bb, tof_params

print("✓ Checkpoint loading functions defined")

## 9. Dataset Classes - Part 1: Inertial and Rotation Datasets

Define PyTorch Dataset classes for accelerometer and rotation (quaternion) sequence data, plus collate functions.


In [ ]:
class InertialSequenceDataset(Dataset):
    """Dataset for accelerometer-only sequences."""
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        acc = torch.tensor(data[["acc_x", "acc_y", "acc_z"]].values, dtype=torch.float32).T
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return acc, label


class RotSequenceDataset(Dataset):
    """Dataset for rotation (quaternion)-only sequences."""
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        rot = torch.tensor(data[ROT_COLS].values.astype(np.float32), dtype=torch.float32).T
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return rot, label


class AccRotSequenceDataset(Dataset):
    """Dataset for accelerometer + rotation (acc+rot fusion) sequences."""
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        acc = torch.tensor(data[["acc_x", "acc_y", "acc_z"]].values, dtype=torch.float32).T
        rot = torch.tensor(data[ROT_COLS].values, dtype=torch.float32).T
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return acc, rot, label


def collate_inertial(batch):
    """Pad acc sequences to same length, transpose to (B, 3, T)."""
    accs, labels = zip(*batch)
    acc_padded = pad_sequence([a.T for a in accs], batch_first=True, padding_value=0).transpose(1, 2)
    return acc_padded, torch.stack(labels)


def collate_rot(batch):
    """Pad rot sequences to same length, transpose to (B, 4, T)."""
    rots, labels = zip(*batch)
    rot_padded = pad_sequence([r.T for r in rots], batch_first=True, padding_value=0).transpose(1, 2)
    return rot_padded, torch.stack(labels)


def collate_acc_rot(batch):
    """Pad acc and rot sequences, return both."""
    accs, rots, labels = zip(*batch)
    acc_padded = pad_sequence([a.T for a in accs], batch_first=True, padding_value=0).transpose(1, 2)
    rot_padded = pad_sequence([r.T for r in rots], batch_first=True, padding_value=0).transpose(1, 2)
    return acc_padded, rot_padded, torch.stack(labels)

print("✓ Dataset classes Part 1 defined")

## 10. Dataset Classes - Part 2: ToF and Triple-Fusion Datasets

Define datasets for Time-of-Flight data and combined (Acc+Rot+ToF) sequences with modality presence flags.


In [ ]:
def _sequence_has_valid_rot(data: pd.DataFrame) -> bool:
    """Check if sequence has non-NaN rotation data."""
    return not data[ROT_COLS].isna().any().any()


def _sequence_has_valid_tof(data: pd.DataFrame, tof_cols) -> bool:
    """Check if sequence has any ToF data (not all -1)."""
    return (data[tof_cols] != -1).any().any()


class ToFSequenceDataset(Dataset):
    """Dataset for Time-of-Flight (depth image) sequences."""
    def __init__(self, dataframe: pd.DataFrame, tof_cols=None):
        self.df = dataframe.reset_index(drop=True)
        self.tof_cols = tof_cols if tof_cols is not None else get_tof_columns(self.df)
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        tof = data[self.tof_cols].values.astype(np.float64)
        tof[tof == -1] = TOF_MAX
        tof[np.isnan(tof)] = TOF_MAX
        tof = np.nan_to_num(tof, nan=TOF_MAX, posinf=TOF_MAX, neginf=TOF_MIN)
        tof_per_frame = _normalize_tof_01(_tof_frame_to_5x8x8(tof))
        x = torch.clamp(torch.tensor(tof_per_frame, dtype=torch.float32), 0.0, 1.0)
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return x, label


def collate_tof(batch):
    """Pad ToF sequences, return lengths for variable-length processing."""
    tof_list, labels = zip(*batch)
    tof_padded = pad_sequence(tof_list, batch_first=True, padding_value=0.0)
    lengths = torch.tensor([t.shape[0] for t in tof_list], dtype=torch.long)
    return tof_padded, lengths, torch.stack(labels)


class AccRotToFSequenceDataset(Dataset):
    """Dataset for Acc + Rot + ToF (triple fusion), tracks modality presence."""
    def __init__(self, dataframe: pd.DataFrame, tof_cols):
        self.df = dataframe.reset_index(drop=True)
        self.tof_cols = tof_cols
        self.sequences = list(self.df.groupby("sequence_id"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _sid, data = self.sequences[idx]
        acc = torch.tensor(data[["acc_x", "acc_y", "acc_z"]].values, dtype=torch.float32).T
        rot_np = data[ROT_COLS].values.T.astype(np.float32)
        rot = torch.tensor(np.nan_to_num(rot_np, nan=0.0), dtype=torch.float32)
        has_rot = torch.tensor(_sequence_has_valid_rot(data), dtype=torch.bool)
        has_tof = torch.tensor(_sequence_has_valid_tof(data, self.tof_cols), dtype=torch.bool)
        
        tof = data[self.tof_cols].values.astype(np.float64)
        tof[tof == -1] = TOF_MAX
        tof[np.isnan(tof)] = TOF_MAX
        tof = np.nan_to_num(tof, nan=TOF_MAX, posinf=TOF_MAX, neginf=0.0)
        tof_per_frame = _tof_frame_to_5x8x8(tof)
        tof_per_frame = _normalize_tof_01(tof_per_frame)
        tof_tensor = torch.clamp(torch.tensor(tof_per_frame, dtype=torch.float32), 0.0, 1.0)
        if not has_tof.item():
            tof_tensor = torch.zeros(1, 5, 8, 8, dtype=torch.float32)
        
        label = torch.tensor(data["gesture_encoded"].iloc[0], dtype=torch.long)
        return acc, rot, tof_tensor, has_rot, has_tof, label


def collate_acc_rot_tof(batch):
    """Pad Acc, Rot, ToF; return lengths and modality presence flags."""
    accs, rots, tofs, has_rots, has_tofs, labels = zip(*batch)
    acc_padded = pad_sequence([a.T for a in accs], batch_first=True, padding_value=0).transpose(1, 2)
    rot_padded = pad_sequence([r.T for r in rots], batch_first=True, padding_value=0).transpose(1, 2)
    tof_padded = pad_sequence(list(tofs), batch_first=True, padding_value=0.0)
    lengths = torch.tensor([t.shape[0] for t in tofs], dtype=torch.long)
    return (
        acc_padded,
        rot_padded,
        tof_padded,
        lengths,
        torch.stack(list(has_rots)),
        torch.stack(list(has_tofs)),
        torch.stack(labels),
    )

print("✓ Dataset classes Part 2 defined")

## 11. Training Loop and Utilities

Generic fitting function and modality-specific train/eval step functions.


In [11]:
def save_model_and_metadata(
    model,
    optimizer,
    gesture_map,
    history,
    train_seq_ids,
    val_seq_ids,
    test_seq_ids,
    scheduler,
    filepath: str,
):
    """Save model checkpoint with metadata and training history."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "gesture_map": gesture_map,
        "history": history,
        "num_classes": len(gesture_map),
        "train_sequence_ids": train_seq_ids,
        "val_sequence_ids": val_seq_ids,
        "test_sequence_ids": test_seq_ids,
    }
    torch.save(checkpoint, filepath)
    print(f"✓ Model checkpoint saved to {filepath}")


def fit_model(
    model,
    train_loader,
    val_loader,
    device,
    criterion,
    optimizer,
    scheduler,
    save_path: str,
    train_step,
    eval_step,
    num_epochs: int = NUM_EPOCHS,
    early_stop_patience: int = EARLY_STOP_PATIENCE,
    grad_clip_params=None,
):
    """Generic training loop with early stopping and LR scheduling."""
    if grad_clip_params is None:
        grad_clip_params = list(model.parameters())
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_acc = 0.0
    patience_counter = 0
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = epoch_acc = 0.0
        n_batches = 0
        for batch in train_loader:
            optimizer.zero_grad()
            loss, batch_acc = train_step(model, batch, device, criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(grad_clip_params, max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            epoch_acc += batch_acc.item()
            n_batches += 1
        avg_train_loss = epoch_loss / n_batches
        avg_train_acc = epoch_acc / n_batches

        model.eval()
        val_loss = val_acc = 0.0
        val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                l, a = eval_step(model, batch, device, criterion)
                val_loss += l
                val_acc += a
                val_batches += 1
        avg_val_loss = val_loss / val_batches
        avg_val_acc = val_acc / val_batches
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc * 100)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc * 100)
        history["lr"].append(current_lr)

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"| Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc*100:.2f}% "
            f"| Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc*100:.2f}% "
            f"| LR: {current_lr:.6f} | Best Val: {best_val_acc*100:.2f}%"
        )
        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs!")
            break

    print(f"\n✓ Training complete. Best validation accuracy: {best_val_acc*100:.2f}%")
    return history


# Train/eval step functions for each modality
def ts_acc(m, batch, device, crit):
    """Train step: accelerometer only."""
    acc, y = batch
    acc = acc.to(device)
    y = y.to(device)
    out = m(acc)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_acc(m, batch, device, crit):
    """Eval step: accelerometer only."""
    acc, y = batch
    acc = acc.to(device)
    y = y.to(device)
    out = m(acc)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()


def ts_rot(m, batch, device, crit):
    """Train step: rotation only."""
    rot, y = batch
    rot = rot.to(device)
    y = y.to(device)
    out = m(rot)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_rot(m, batch, device, crit):
    """Eval step: rotation only."""
    rot, y = batch
    rot = rot.to(device)
    y = y.to(device)
    out = m(rot)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()


def ts_fusion_ar(m, batch, device, crit):
    """Train step: acc+rot fusion."""
    acc, rot, y = batch
    acc = acc.to(device)
    rot = rot.to(device)
    y = y.to(device)
    out = m(acc, rot)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_fusion_ar(m, batch, device, crit):
    """Eval step: acc+rot fusion."""
    acc, rot, y = batch
    acc = acc.to(device)
    rot = rot.to(device)
    y = y.to(device)
    out = m(acc, rot)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()


def ts_tof(m, batch, device, crit):
    """Train step: ToF only."""
    tof_padded, lengths, y = batch
    tof_padded = tof_padded.to(device)
    lengths = lengths.to(device)
    y = y.to(device)
    out = m(tof_padded, lengths)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_tof(m, batch, device, crit):
    """Eval step: ToF only."""
    tof_padded, lengths, y = batch
    tof_padded = tof_padded.to(device)
    lengths = lengths.to(device)
    y = y.to(device)
    out = m(tof_padded, lengths)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()


def ts_fusion_tof(m, batch, device, crit):
    """Train step: triple fusion (acc+rot+tof)."""
    acc, rot, tof_padded, lengths, has_rot, has_tof, y = batch
    acc = acc.to(device)
    rot = rot.to(device)
    tof_padded = tof_padded.to(device)
    lengths = lengths.to(device)
    has_rot = has_rot.to(device)
    has_tof = has_tof.to(device)
    y = y.to(device)
    out = m(acc, rot, tof_padded, lengths, has_rot, has_tof)
    return crit(out, y), (out.argmax(1) == y).float().mean()


def es_fusion_tof(m, batch, device, crit):
    """Eval step: triple fusion (acc+rot+tof)."""
    acc, rot, tof_padded, lengths, has_rot, has_tof, y = batch
    acc = acc.to(device)
    rot = rot.to(device)
    tof_padded = tof_padded.to(device)
    lengths = lengths.to(device)
    has_rot = has_rot.to(device)
    has_tof = has_tof.to(device)
    y = y.to(device)
    out = m(acc, rot, tof_padded, lengths, has_rot, has_tof)
    return crit(out, y).item(), (out.argmax(1) == y).float().mean().item()

print("✓ Training loop and step functions defined")

✓ Training loop and step functions defined


## 12. Data Preparation Helper

Load, split, and encode labels in a single function call for beginning each pipeline stage.


In [8]:
def load_split_data_encoded():
    """Load CSV, add BFRB flag, split into train/val/test, encode labels."""
    train_df = load_train_data()
    train_df["is_bfrb"] = train_df["gesture"].isin(BFRB_GESTURES)
    trainset_df, valset_df, testset_df = final_robust_split(train_df)
    trainset_df = trainset_df.reset_index(drop=True)
    valset_df = valset_df.reset_index(drop=True)
    testset_df = testset_df.reset_index(drop=True)
    return apply_label_encoding(train_df, trainset_df, valset_df, testset_df)

print("✓ load_split_data_encoded() defined")

# Perform data splitting and encoding immediately
train_df_full, trainset_df, valset_df, testset_df, le, gesture_map = load_split_data_encoded()
print(f"✓ Data split: train {len(trainset_df)} rows ({trainset_df['sequence_id'].nunique()} seq), "
      f"val {len(valset_df)} ({valset_df['sequence_id'].nunique()} seq), "
      f"test {len(testset_df)} ({testset_df['sequence_id'].nunique()} seq)")

✓ load_split_data_encoded() defined
✓ Loaded train.csv from ./data


/var/folders/80/npxmtc2j0qjbb4wm1sm23l_00000gp/T/ipykernel_44521/2956283598.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["is_bfrb"] = train_df["gesture"].isin(BFRB_GESTURES)


✓ Data split: train 406777 rows (5762 seq), val 79383 (1170 seq), test 88785 (1219 seq)


## 13. Pipeline Stage 1: Train Accelerometer-Only CNN

Load data, create accelerometer dataset, train and save Multistream_CNN_acc_only.pth


In [ ]:
def stage_acc_multistream(device: torch.device):
    """Stage 1: Train accelerometer-only CNN."""
    # Use pre-loaded and split data
    print(f"Split: train {len(trainset_df)} rows ({trainset_df['sequence_id'].nunique()} seq), "
          f"val {len(valset_df)} ({valset_df['sequence_id'].nunique()} seq), "
          f"test {len(testset_df)} ({testset_df['sequence_id'].nunique()} seq)")
    n_cls = len(le.classes_)
    train_seq_ids = trainset_df["sequence_id"].unique().tolist()
    val_seq_ids = valset_df["sequence_id"].unique().tolist()
    test_seq_ids = testset_df["sequence_id"].unique().tolist()
    
    tl = DataLoader(
        InertialSequenceDataset(trainset_df),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_inertial,
    )
    vl = DataLoader(
        InertialSequenceDataset(valset_df),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_inertial,
    )
    
    model = MultistreamCNNInertialNet(num_classes=n_cls).to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    
    hist = fit_model(model, tl, vl, device, crit, opt, sch, PATH_ACC, ts_acc, es_acc)
    load_state_dict_into_model(model, PATH_ACC, device)
    save_model_and_metadata(model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_ACC)
    
    return model, hist, le

print("✓ stage_acc_multistream() defined")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    acc_model, acc_hist, acc_le = stage_acc_multistream(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 14. Pipeline Stage 2: Train Rotation-Only CNN

Clean rotation data, create rotation dataset, train and save Multistream_CNN_rot_only.pth


In [ ]:
def stage_rot_multistream(device: torch.device):
    """Stage 2: Train rotation-only CNN (drop sequences with missing quaternions)."""
    # Use pre-loaded data, apply quaternion cleaning
    trainset_df_clean, train_seq_ids = drop_sequences_with_missing_quaternions(trainset_df)
    valset_df_clean, val_seq_ids = drop_sequences_with_missing_quaternions(valset_df)
    testset_df_clean, test_seq_ids = drop_sequences_with_missing_quaternions(testset_df)
    n_cls = len(le.classes_)
    
    tl = DataLoader(
        RotSequenceDataset(trainset_df_clean),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_rot,
    )
    vl = DataLoader(
        RotSequenceDataset(valset_df_clean),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_rot,
    )
    
    model = MultistreamCNNRotNet(num_classes=n_cls).to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    
    hist = fit_model(model, tl, vl, device, crit, opt, sch, PATH_ROT, ts_rot, es_rot)
    load_state_dict_into_model(model, PATH_ROT, device)
    save_model_and_metadata(model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_ROT)
    
    return model, hist, le

print("✓ stage_rot_multistream() defined")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    rot_model, rot_hist, rot_le = stage_rot_multistream(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 15. Pipeline Stage 3: Train Acc+Rot Fusion CNN

Fuse accelerometer and rotation models, train new fusion head, save Fusion_Multistream_CNN.pth


In [ ]:
def stage_fusion_acc_rot(device: torch.device):
    """Stage 3: Fuse pre-trained acc and rot models, train fusion head."""
    # Use pre-loaded data, apply quaternion cleaning
    trainset_df_clean, train_seq_ids = drop_sequences_with_missing_quaternions(trainset_df)
    valset_df_clean, val_seq_ids = drop_sequences_with_missing_quaternions(valset_df)
    testset_df_clean, test_seq_ids = drop_sequences_with_missing_quaternions(testset_df)
    n_cls = len(le.classes_)
    
    tl = DataLoader(
        AccRotSequenceDataset(trainset_df_clean),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_acc_rot,
    )
    vl = DataLoader(
        AccRotSequenceDataset(valset_df_clean),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_acc_rot,
    )
    
    acc_bb = make_acc_feature_extractor(n_cls, PATH_ACC, device)
    rot_bb = make_rot_feature_extractor(n_cls, PATH_ROT, device)
    model = FusionMultistreamCNN(acc_bb, rot_bb, num_classes=n_cls, hidden_dim=64).to(device)
    
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    
    hist = fit_model(
        model, tl, vl, device, crit, opt, sch, PATH_FUSION, ts_fusion_ar, es_fusion_ar, grad_clip_params=trainable
    )
    load_state_dict_into_model(model, PATH_FUSION, device)
    save_model_and_metadata(
        model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_FUSION
    )
    
    return model, hist, le

print("✓ stage_fusion_acc_rot() defined")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    fusion_model, fusion_hist, fusion_le = stage_fusion_acc_rot(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 16. Pipeline Stage 4: Train ToF CNN

Clean ToF data, create 2D depth image dataset, train and save ToF_CNN.pth (skipped if file exists)


In [ ]:
def stage_tof_cnn(device: torch.device):
    """Stage 4: Train ToF CNN (drop sequences with missing ToF data)."""
    # Use pre-loaded data, apply ToF cleaning
    tof_cols = get_tof_columns(train_df_full)
    trainset_df_clean, train_seq_ids = drop_sequences_with_missing_tof(trainset_df, tof_cols)
    valset_df_clean, val_seq_ids = drop_sequences_with_missing_tof(valset_df, tof_cols)
    testset_df_clean, test_seq_ids = drop_sequences_with_missing_tof(testset_df, tof_cols)
    n_cls = len(le.classes_)
    
    tl = DataLoader(
        ToFSequenceDataset(trainset_df_clean, tof_cols),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_tof,
    )
    vl = DataLoader(
        ToFSequenceDataset(valset_df_clean, tof_cols),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_tof,
    )
    
    model = ToFCNN(num_classes=n_cls).to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    
    hist = fit_model(model, tl, vl, device, crit, opt, sch, PATH_TOF, ts_tof, es_tof)
    ckpt = torch.load(PATH_TOF, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    
    save_model_and_metadata(
        model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_TOF
    )
    
    return model, hist, le

print("✓ stage_tof_cnn() defined")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    tof_model, tof_hist, tof_le = stage_tof_cnn(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 17. Pipeline Stage 5: Fine-tune Fusion+ToF Deep Model

Combine all three modalities (Acc+Rot+ToF) with missing modality handling, train final model, save Fusion_ToF_Multistream_CNN_finetune.pth


In [ ]:
def stage_fusion_tof_finetune(device: torch.device):
    """Stage 5: Fine-tune triple fusion (Acc+Rot+ToF with missing modality handling)."""
    # Use pre-loaded data
    print(
        f"Split (no sensor drop): train {len(trainset_df)} rows / {trainset_df['sequence_id'].nunique()} seq, "
        f"val {len(valset_df)} / {valset_df['sequence_id'].nunique()}, "
        f"test {len(testset_df)} / {testset_df['sequence_id'].nunique()}"
    )
    
    tof_cols = get_tof_columns(train_df)
    compute_sequence_sensor_stats(trainset_df, tof_cols, "TRAIN")
    compute_sequence_sensor_stats(valset_df, tof_cols, "VAL")
    compute_sequence_sensor_stats(testset_df, tof_cols, "TEST")
    
    train_seq_ids = trainset_df["sequence_id"].unique().tolist()
    val_seq_ids = valset_df["sequence_id"].unique().tolist()
    test_seq_ids = testset_df["sequence_id"].unique().tolist()
    n_cls = len(le.classes_)
    
    tl = DataLoader(
        AccRotToFSequenceDataset(trainset_df, tof_cols),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_acc_rot_tof,
    )
    vl = DataLoader(
        AccRotToFSequenceDataset(valset_df, tof_cols),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_acc_rot_tof,
    )
    
    # Load pre-trained components
    fusion_128 = make_fusion_finetune_128(n_cls, PATH_FUSION, device)
    tof_feat = make_tof_finetune_extractor(n_cls, PATH_TOF, device)
    model = ToFFusionMultistreamCNNDeepFinetune(fusion_128, tof_feat, num_classes=n_cls).to(device)
    
    # Attempt warm-start from triple fusion checkpoint if it exists
    warm_path = Path(PATH_TRIPLE_WARMSTART)
    if warm_path.is_file():
        warm = torch.load(warm_path, map_location=device, weights_only=False)
        warm_sd = warm["model_state_dict"] if isinstance(warm, dict) and "model_state_dict" in warm else warm
        loaded, skipped, missing = load_state_dict_shape_safe(model, warm_sd)
        print(f"Warm-start: loaded {len(loaded)}, skipped shape {len(skipped)}, missing {len(missing)}")
    else:
        print(f"No warm-start at {warm_path}")
    
    # Collect parameter groups: head, fusion backbone, ToF backbone
    head_p, fusion_p, tof_p = collect_param_groups(model)
    opt = optim.AdamW(
        [
            {"params": head_p, "lr": LR_HEAD_FT, "weight_decay": WEIGHT_DECAY_FT},
            {"params": fusion_p, "lr": LR_BACKBONE_FT, "weight_decay": WEIGHT_DECAY_FT},
            {"params": tof_p, "lr": LR_BACKBONE_FT, "weight_decay": WEIGHT_DECAY_FT},
        ]
    )
    crit = nn.CrossEntropyLoss()
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    
    hist = fit_model(
        model,
        tl,
        vl,
        device,
        crit,
        opt,
        sch,
        PATH_FINETUNE,
        ts_fusion_tof,
        es_fusion_tof,
        grad_clip_params=trainable,
    )
    
    ckpt_best = torch.load(PATH_FINETUNE, map_location=device, weights_only=False)
    model.load_state_dict(ckpt_best)
    save_model_and_metadata(
        model, opt, gesture_map, hist, train_seq_ids, val_seq_ids, test_seq_ids, sch, PATH_FINETUNE
    )
    print(f"✓ Saved fine-tuned checkpoint to {PATH_FINETUNE}")
    
    return model, hist, le

print("✓ stage_fusion_tof_finetune() defined")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    finetune_model, finetune_hist, finetune_le = stage_fusion_tof_finetune(device)

## 18. Main Pipeline Orchestration

Coordinate all 5 stages: Acc CNN → Rot CNN → Fusion → ToF → Fine-tune, with cleanup between stages.


In [ ]:
def run_full_training_pipeline(skip_tof_if_exists: bool = True):
    """
    Run all 5 training stages sequentially with garbage collection between stages.
    
    Returns dict with keys: 'acc', 'rot', 'fusion_ar', 'tof', 'finetune'
    - 'tof' is omitted if skip_tof_if_exists=True and file exists
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Device: {device}\n")
    
    out = {}
    
    print("=" * 80)
    print("STAGE 1: Accelerometer-only CNN")
    print("=" * 80)
    out["acc"] = stage_acc_multistream(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\n" + "=" * 80)
    print("STAGE 2: Rotation-only CNN")
    print("=" * 80)
    out["rot"] = stage_rot_multistream(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\n" + "=" * 80)
    print("STAGE 3: Acc+Rot Fusion CNN")
    print("=" * 80)
    out["fusion_ar"] = stage_fusion_acc_rot(device)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\n" + "=" * 80)
    print("STAGE 4: Time-of-Flight CNN")
    print("=" * 80)
    tof_p = Path(PATH_TOF)
    if skip_tof_if_exists and tof_p.is_file():
        print(f"⏭️  Skipping ToF training; already exists: {tof_p}")
    else:
        out["tof"] = stage_tof_cnn(device)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print("\n" + "=" * 80)
    print("STAGE 5: Fine-tune Fusion+ToF Deep Model")
    print("=" * 80)
    out["finetune"] = stage_fusion_tof_finetune(device)
    
    print("\n" + "=" * 80)
    print("✓ PIPELINE COMPLETE")
    print("=" * 80)
    print(f"Keys: {list(out.keys())}")
    return out

print("✓ run_full_training_pipeline() defined")